# Safety eval on the JupyterHub A100 box

Open this notebook **inside** the hub, from the cloned repo. It does the same thing
`./run.sh` does from a terminal — start here if you would rather see the checks in
cells than in a log.

**Run the cells in order.** Cell 1 pins the GPUs, and pinning only works *before*
anything imports `torch`. If you have already imported torch in this kernel,
restart it (Kernel → Restart) before running cell 1 — otherwise
`CUDA_VISIBLE_DEVICES` is set and silently ignored, and the run lands on whatever
cards happen to be free, which on a shared box means somebody else's.

## 1. Pin the GPUs (must be the first cell you run)

Reads `SAFETY_GPUS` from `.env` (we agreed on **4,5**), refuses to start if either
card is already busy, and prints the whole board so you can see who is where.

In [ ]:
import sys
from pathlib import Path

HUB_DIR = Path.cwd() if Path.cwd().name == "jupyterhub" else Path.cwd() / "safety" / "jupyterhub"
REPO_ROOT = HUB_DIR.parent.parent
for p in (str(HUB_DIR), str(REPO_ROOT / "commitment"), str(REPO_ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)

import gpu  # noqa: E402  — after the path setup above, before torch

selection = gpu.pin_from_env()

## 2. Confirm torch is on those exact cards

The count alone is not proof: CUDA can enumerate devices in a different order than
`nvidia-smi`, so this compares **UUIDs**.

In [ ]:
import torch  # noqa: F401  — first torch import, AFTER the pin

assert gpu.verify(selection)["verified"], "not on the requested GPUs"

## 3. Preflight

Imports, the dataset registry, `ACT_LAYERS` in range for this model, HF cache room.
Seconds, no weights loaded. Equivalent to `./run.sh preflight`.

In [ ]:
!cd {REPO_ROOT} && python3 safety/jupyterhub/preflight.py

## 4. The smoke ladder

Eight rungs, cheapest first: datasets → prompts → matcher → unit tests → preflight
→ batch equivalence (GPU) → one full trace (GPU) → curve sanity.

Drop `--skip-gpu` once the CPU rungs pass to run the two GPU rungs as well.

In [ ]:
!cd {REPO_ROOT} && python3 safety/jupyterhub/smoke.py --skip-gpu

## 5. A real run

Surface pass, batched, heuristic-scored. `--judge` adds the HarmBench-13B pass that
is the scorer of record for the 1.0b number — the heuristic is for plumbing only.

The cost knob is `--rollouts`: 0 is the surface sweep; `k` adds `prefixes * k` extra
generations **per trace**.

In [ ]:
!cd {REPO_ROOT} && python3 safety/jupyterhub/run_local.py collect \
    --n 12 --budgets natural --samples 1 --batch-size 8 --run-name nb-demo

## 6. Read the rows

Always look at actual traces before believing a rate. The heuristic scorer has been
measured wrong on a large fraction of rows; a number with no trace behind it is not
a result yet.

In [ ]:
import json

run_dir = REPO_ROOT / "safety" / "outputs" / "nb-demo"
rows = [json.loads(line) for line in open(run_dir / "traces.jsonl")]
print(json.dumps(json.load(open(run_dir / "summary.json")), indent=2))

for r in rows[:3]:
    answer = r["response_text"].split("</think>", 1)[-1].strip()
    print(f"\n--- {r['prompt_id']} budget={r['budget_condition']} "
          f"think={r['thinking_tokens']} outcome={r['outcome']}")
    print("   ", answer[:300])